In [ ]:
import os 
os.getcwd()
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
import os

In [ ]:

# Load the filez
data = loadmat('data.mat')
illumination = loadmat("illumination.mat")
pose=loadmat("pose.mat")
# See all the keys (variables stored in the .mat file)
print(f"data keys are {data.keys()}")
print(f"illumination keys are {illumination.keys()}")
print(f"pose keys are {pose.keys()}")

In [ ]:
n=np.arange(1,201,1)
data_classified = {}
for i in n:
    imgs=[]
    for j in range(3):
        imgs.append(data["face"][:,:,3*i-3+j])
    data_classified[i]=imgs

In [ ]:
n_2=np.arange(0,68,1)
n_2p=np.arange(0,13,1)
pose_classified={}
for i in n_2:
    imgs=[]
    for j in n_2p:
        imgs.append(pose["pose"][:,:,j,i])
    pose_classified[i]=imgs

In [ ]:
n_3=np.arange(0,68,1)
n_3i=np.arange(0,21,1)
illum_classified={}
for i in n_3:
    imgs=[]
    for j in n_3i:
        imgs.append(illumination["illum"][:,j,i])
    illum_classified[i]=imgs

# data + labeling

In [ ]:
flattened_data = np.array([data["face"][:, :, i].flatten() for i in range(600)])
person_labels = np.repeat(np.arange(200), 3) #for task1

# for task2:
neutral_indices = list(range(0, 600, 3))
expression_indices = list(range(1, 600, 3))
binary_labels = np.zeros(600, dtype=int)
binary_labels[expression_indices] = 1


# PCA_funciton

In [ ]:
import numpy as np

def compute_pca(data_matrix, num_components=None, variance_threshold=None):
    """
    Perform PCA on a data matrix (samples × features).
    
    Parameters:
        data_matrix: np.ndarray, shape (n_samples, n_features)
        num_components: int or None — number of components to keep
        variance_threshold: float or None — keep components that explain up to this cumulative variance (e.g., 0.95)
    
    Returns:
        pca_result: projected data, shape (n_samples, num_components)
        components: principal components (eigenvectors)
        explained_variance_ratio: array of variance explained by each component
        mean: mean of original data (for inverse transform if needed)
    """
    # Step 1: Center the data
    mean = np.mean(data_matrix, axis=0)
    centered_data = data_matrix - mean

    # Step 2: Covariance matrix
    cov_matrix = np.cov(centered_data, rowvar=False)

    # Step 3: Eigen decomposition
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

    # Step 4: Sort in descending order
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    # Step 5: Compute explained variance
    explained_variance_ratio = eigenvalues / np.sum(eigenvalues)
    cumulative_variance = np.cumsum(explained_variance_ratio)

    # Step 6: Determine number of components
    if variance_threshold is not None:
        num_components = np.argmax(cumulative_variance >= variance_threshold) + 1
        print(f"Using {num_components} components to explain {variance_threshold*100:.1f}% variance.")
    elif num_components is None:
        num_components = data_matrix.shape[1]  # Keep all

    # Step 7: Select components and project
    selected_components = eigenvectors[:, :num_components]
    pca_result = np.dot(centered_data, selected_components)

    return pca_result, selected_components, explained_variance_ratio[:num_components], mean


# MDA Function

In [ ]:
import numpy as np

def compute_mda(data_matrix, labels, num_components=None):
    """
    Perform MDA (also known as LDA) on the given data.

    Parameters:
        data_matrix: np.ndarray of shape (n_samples, n_features)
        labels: array-like of shape (n_samples,)
        num_components: int or None — number of components to retain (must be ≤ n_classes - 1)

    Returns:
        mda_result: Projected data of shape (n_samples, num_components)
        components: Eigenvectors used for projection (n_features, num_components)
        eigenvalues: Corresponding eigenvalues
        overall_mean: Mean of the original data
    """
    n_samples, n_features = data_matrix.shape
    unique_classes = np.unique(labels)
    n_classes = len(unique_classes)

    # Step 1: Center the data
    overall_mean = np.mean(data_matrix, axis=0)
    centered_data = data_matrix - overall_mean

    # Step 2: Compute class means
    class_means = []
    for c in unique_classes:
        class_data = data_matrix[labels == c]
        class_mean = np.mean(class_data, axis=0)
        class_means.append(class_mean)

    # Step 3: Compute between-class scatter matrix (S_B)
    S_B = np.zeros((n_features, n_features))
    for i, c in enumerate(unique_classes):
        n_i = np.sum(labels == c)
        mean_diff = (class_means[i] - overall_mean).reshape(-1, 1)
        S_B += (n_i / n_samples) * (mean_diff @ mean_diff.T)

    # Step 4: Compute within-class scatter matrix (S_W)
    S_W = np.zeros((n_features, n_features))
    for i, c in enumerate(unique_classes):
        class_data = data_matrix[labels == c]
        n_i = class_data.shape[0]
        class_centered = class_data - class_means[i]
        S_W += (n_i / n_samples) * (class_centered.T @ class_centered) / n_i

    # Step 5: Solve generalized eigenvalue problem
    S_W_inv = np.linalg.pinv(S_W)
    eig_matrix = S_W_inv @ S_B
    eigenvalues, eigenvectors = np.linalg.eigh(eig_matrix)

    # Step 6: Sort eigenvectors by eigenvalue magnitude (descending)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    # Step 7: Limit number of components
    max_components = n_classes - 1
    if num_components is None or num_components > max_components:
        num_components = max_components
        print(f"Using max possible components for MDA: {num_components}")

    selected_components = eigenvectors[:, :num_components]
    mda_result = centered_data @ selected_components

    return mda_result, selected_components, eigenvalues[:num_components], overall_mean


# data seperation

In [ ]:
def separate_train_test_manual(data, labels, sub_num_total, train_num, task_num, random_state=42):
    np.random.seed(random_state)
    
    if task_num == 1:  # Person identification with all 3 images
        selected_subjects = np.random.choice(sub_num_total, train_num, replace=False)
        train_indices = []
        test_indices = []

        # For each subject, put 2 images in training and 1 in testing
        for s in selected_subjects:
            base = 3 * s
            train_indices.extend([base, base + 1])  # First 2 images to training
            test_indices.append(base + 2)           # Last image to testing

        train_set = data[train_indices]
        train_labels = labels[train_indices]
        test_set = data[test_indices]
        test_labels = labels[test_indices]

    elif task_num == 2:
        # Get indices for neutral and expression images
        neutral_indices = list(range(0, 3 * sub_num_total, 3))     # Images at positions 0, 3, 6, ...
        expression_indices = list(range(1, 3 * sub_num_total, 3))  # Images at positions 1, 4, 7, ...
        
        # Create binary labels (0 for neutral, 1 for expression)
        neutral_labels = np.zeros(len(neutral_indices), dtype=int)
        expression_labels = np.ones(len(expression_indices), dtype=int)
        
        # Shuffle each set of indices separately
        np.random.shuffle(neutral_indices)
        np.random.shuffle(expression_indices)
        
        # Split each class into training (80%) and testing (20%)
        neutral_split = int(len(neutral_indices) * 0.8)
        expression_split = int(len(expression_indices) * 0.8)
        
        # Create training and testing sets for each class
        neutral_train = neutral_indices[:neutral_split]
        neutral_test = neutral_indices[neutral_split:]
        expression_train = expression_indices[:expression_split]
        expression_test = expression_indices[expression_split:]
        
        # Combine indices and labels
        train_indices = np.concatenate([neutral_train, expression_train])
        test_indices = np.concatenate([neutral_test, expression_test])
        
        # Create labels matching the indices
        train_labels = np.concatenate([np.zeros(len(neutral_train)), np.ones(len(expression_train))])
        test_labels = np.concatenate([np.zeros(len(neutral_test)), np.ones(len(expression_test))])
        
        # Extract the data
        train_set = data[train_indices]
        test_set = data[test_indices]
    
    else:
        raise ValueError("❌ Invalid task_num. Use 1 for person ID or 2 for expression classification.")
    return train_set, train_labels, test_set, test_labels


# kernel svm gradient descent

In [ ]:
def rbf_kernel(x, y, sigma=1.0):
    return np.exp(-np.linalg.norm(x - y) ** 2 / (2 * sigma ** 2))

def poly_kernel(x, y, degree=3):
    return (np.dot(x, y) + 1) ** degree

def poly_kernel_reverse(x, y, degree=3):
    return (np.dot(x, y) + 1) ** (1/degree)

def linear_kernel (x,y):
    return np.dot (x,y)
class KernelSVM_GD:
    def __init__(self, C=1.0, kernel='rbf', sigma=1.0, degree=3, lr=0.001, max_iter=1000):
        self.C = C
        self.sigma = sigma
        self.degree = degree
        self.kernel_type = kernel
        self.lr = lr
        self.max_iter = max_iter
        self.kernel = self._get_kernel(kernel)

    def _get_kernel(self, kernel_type):
        if kernel_type == 'rbf':
            return lambda x, y: rbf_kernel(x, y, self.sigma)
        elif kernel_type == 'poly':
            return lambda x, y: poly_kernel(x, y, self.degree)
        elif kernel_type == 'poly_rev':
            return lambda x, y: poly_kernel_reverse(x, y, self.degree)
        elif kernel_type == 'linear':
            return lambda x, y: linear_kernel(x, y)
        else:
            raise ValueError("Unsupported kernel.")

    def fit(self, X, y):
        n_samples = X.shape[0]
        self.X = X
        self.y = y
        self.alphas = np.zeros(n_samples)

        # Precompute full kernel matrix
        K = np.zeros((n_samples, n_samples))
        for i in range(n_samples):
            for j in range(n_samples):
                K[i, j] = self.kernel(X[i], X[j])

        # Dual objective: maximize L = sum αi - 1/2 sum_i,j αi αj yi yj K(xi, xj)
        for it in range(self.max_iter):
            for i in range(n_samples):
                # Gradient of dual L wrt α_i
                gradient = 1 - np.sum(
                    self.alphas * y * y[i] * K[:, i]
                )
                self.alphas[i] += self.lr * gradient

                # Project α back into bounds [0, C]
                self.alphas[i] = np.clip(self.alphas[i], 0, self.C)

        # Support vectors
        sv = self.alphas > 1e-5
        self.support_vectors = X[sv]
        self.support_vector_labels = y[sv]
        self.alphas = self.alphas[sv]

        # Bias: average over support vectors
        self.b = np.mean([
            y_i - np.sum(self.alphas * self.support_vector_labels *
                         np.array([self.kernel(x_i, x_j) for x_j in self.support_vectors]))
            for x_i, y_i in zip(self.support_vectors, self.support_vector_labels)
        ])

    def project(self, X):
        y_pred = []
        for x in X:
            result = np.sum([
                a * y_sv * self.kernel(x, x_sv)
                for a, y_sv, x_sv in zip(self.alphas, self.support_vector_labels, self.support_vectors)
            ])
            y_pred.append(result + self.b)
        return np.array(y_pred)

    def predict(self, X):
        return np.sign(self.project(X))


# adaboost with kernel svm with gradient descent

In [ ]:
class AdaBoostWithSVM_GD:
    def __init__(self, n_estimators=30, C=1.0, kernel='linear', sigma=1.0, degree=3,
                 lr=0.001, max_iter=500):
        self.n_estimators = n_estimators
        self.C = C
        self.kernel = kernel
        self.sigma = sigma
        self.degree = degree
        self.lr = lr
        self.max_iter = max_iter
        self.models = []
        self.alphas = []

        self.train_acc_history = []
        self.test_acc_history = []
        self.X_test = None
        self.y_test = None

    def fit(self, X, y, X_test=None, y_test=None):
        self.X_test = X_test
        self.y_test = y_test

        n_samples = X.shape[0]
        w = np.ones(n_samples) / n_samples

        for t in range(self.n_estimators):
#             if t == 0:
#                 indices = np.arange(n_samples)
#             else:
            indices = np.random.choice(np.arange(n_samples), size=n_samples, replace=True, p=w)

            X_sample = X[indices]
            y_sample = y[indices]

            # Train a weak learner
            clf = KernelSVM_GD(C=self.C, kernel=self.kernel, sigma=self.sigma,
                               degree=self.degree, lr=self.lr, max_iter=self.max_iter)
            clf.fit(X_sample, y_sample)

            # Predict on training set
            y_pred = clf.predict(X)
            incorrect = (y_pred != y).astype(float)
            epsilon = np.sum(w * incorrect)

            if epsilon <= 1e-10 or epsilon >= 0.5:
                continue

            alpha = 0.5 * np.log((1 - epsilon) / epsilon)
            self.models.append(clf)
            self.alphas.append(alpha)

            # Update weights
            w *= np.exp(-alpha * y * y_pred)
            w /= np.sum(w)

            # Accuracy tracking
            train_pred = self.predict(X)
            train_acc = np.mean(train_pred == y)
            self.train_acc_history.append(train_acc)

            if X_test is not None and y_test is not None:
                test_pred = self.predict(X_test)
                test_acc = np.mean(test_pred == y_test)
                self.test_acc_history.append(test_acc)

    def predict(self, X):
        final_pred = np.zeros(X.shape[0])
        for alpha, model in zip(self.alphas, self.models):
            final_pred += alpha * model.predict(X)
        return np.sign(final_pred)


In [ ]:
# ====== STEP 1: Prepare Data ======
train_set, train_labels, test_set, test_labels = separate_train_test_manual(
    data=flattened_data,
    labels=person_labels,
    sub_num_total=200,
    train_num=200,
    task_num=2
)

# ====== STEP 2: Convert Labels to {-1, +1} ======
unique_labels = np.unique(train_labels)
assert len(unique_labels) == 2, "Expected binary classification labels."

label_map = {unique_labels[0]: -1, unique_labels[1]: +1}
train_labels_bin = np.vectorize(label_map.get)(train_labels)
test_labels_bin = np.vectorize(label_map.get)(test_labels)


In [ ]:
# ====== STEP 3: Boosted SVM with Gradient Descent-Based SVM ======
boosted_gd = AdaBoostWithSVM_GD(
    n_estimators=20,
    C=1.0,
    kernel='linear',     # can be 'linear', 'poly', etc.
    lr=0.001,
    max_iter=500
)

boosted_gd.fit(train_set, train_labels_bin, X_test=test_set, y_test=test_labels_bin)


In [ ]:
# ====== STEP 4: Evaluate on Train/Test ======
train_pred = boosted_gd.predict(train_set)
test_pred = boosted_gd.predict(test_set)

train_acc = np.mean(train_pred == train_labels_bin) * 100
test_acc = np.mean(test_pred == test_labels_bin) * 100

print(f"Train Accuracy: {train_acc:.2f}%")
print(f"Test Accuracy:  {test_acc:.2f}%")
print(f"Boosting completed with {len(boosted_gd.models)} rounds.")


In [ ]:
# ====== STEP 5: Plot Accuracy over Boosting Rounds ======
import matplotlib.pyplot as plt
import numpy as np

rounds = np.arange(1, len(boosted_gd.train_acc_history) + 1)

plt.plot(rounds, np.array(boosted_gd.train_acc_history) * 100, label="Train Accuracy", marker='o')
plt.plot(rounds, np.array(boosted_gd.test_acc_history) * 100, label="Test Accuracy", marker='s')

plt.xlabel("Boosting Round")
plt.ylabel("Accuracy (%)")
plt.title("AdaBoost with SVM (Gradient Descent): Accuracy over Rounds")
plt.legend()
plt.grid(True)
plt.show()


# MDA + PCA

In [ ]:
# ====== STEP 1: Prepare Data ======
train_set, train_labels, test_set, test_labels = separate_train_test_manual(
    data=flattened_data,
    labels=person_labels,
    sub_num_total=200,
    train_num=200,
    task_num=2
)

# ====== STEP 2: Convert Labels to {-1, +1} ======
unique_labels = np.unique(train_labels)
assert len(unique_labels) == 2, "Expected binary classification labels."

label_map = {unique_labels[0]: -1, unique_labels[1]: +1}
train_labels_bin = np.vectorize(label_map.get)(train_labels)
test_labels_bin = np.vectorize(label_map.get)(test_labels)


In [ ]:


pca_components_list = [1, 2, 3, 10, 50, 80, 100, 120, 139]
pca_adaboost_accuracies = []

for num_components in pca_components_list:
    # Step 1: Compute PCA
    pca_result_train, components, _, mean_vec = compute_pca(train_set, num_components)
    centered_test = test_set - mean_vec
    pca_result_test = np.dot(centered_test, components)

    # Step 2: Train AdaBoost + SVM_GD on PCA-reduced data
    boosted = AdaBoostWithSVM_GD(
        n_estimators=20,
        C=1.0,
        kernel='linear',
        lr=0.001,
        max_iter=500
    )
    boosted.fit(pca_result_train, train_labels_bin, X_test=pca_result_test, y_test=test_labels_bin)

    # Step 3: Get final accuracy
    test_pred = boosted.predict(pca_result_test)
    test_acc = np.mean(test_pred == test_labels_bin) * 100
    pca_adaboost_accuracies.append(test_acc)

    print(f"PCA={num_components} → Test Accuracy: {test_acc:.2f}%")


In [ ]:
mda_components_list = [1]
mda_adaboost_accuracies = []

for num_components in mda_components_list:
    mda_result_train, components, _, mean_vec = compute_mda(train_set, train_labels_bin, num_components)
    centered_test = test_set - mean_vec
    mda_result_test = np.dot(centered_test, components)

    boosted = AdaBoostWithSVM_GD(
        n_estimators=20,
        C=0.8,
        kernel='linear',
        lr=0.01,
        max_iter=2
    )
    boosted.fit(mda_result_train, train_labels_bin, X_test=mda_result_test, y_test=test_labels_bin)

    test_pred = boosted.predict(mda_result_test)
    test_acc = np.mean(test_pred == test_labels_bin) * 100
    mda_adaboost_accuracies.append(test_acc)

    print(f"MDA={num_components} → Test Accuracy: {test_acc:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(pca_components_list, pca_adaboost_accuracies, marker='o', label='PCA + AdaBoost')
plt.axhline(mda_adaboost_accuracies[0], linestyle='--', color="orange",  label=f'MDA + AdaBoost ({mda_adaboost_accuracies[0]})')

plt.xlabel("Number of Components")
plt.ylabel("Test Accuracy (%)")
plt.title("AdaBoost + Kernel SVM (GD) with PCA and MDA")
plt.grid(True)
plt.legend()
plt.show()


# In the report, I only included the results from SVMs trained using gradient descent. However, just for your information, I also tested AdaBoost with SVM using CVXOPT. I didn’t include those results in the report simply because it was already too long. Apologies!

# Kernel SVM cvxopt

In [ ]:
import numpy as np
from cvxopt import matrix, solvers

# -----------------------------
# Kernels
# -----------------------------

def rbf_kernel(x, y, sigma=1.0):
    return np.exp(-np.linalg.norm(x - y) ** 2 / (2 * sigma ** 2))

def poly_kernel(x, y, degree=3):
    return (np.dot(x, y) + 1) ** degree

def poly_kernel_reverse(x, y, degree=3):
    return (np.dot(x, y) + 1) ** (1/degree)

def linear_kernel (x,y):
    return np.dot (x,y)

# -----------------------------
# Kernel SVM Classifier (with optional C)
# -----------------------------

class KernelSVM:
    def __init__(self, C=None, kernel='rbf', sigma=1.0, degree=3):
        self.C = C  # None means hard-margin
        self.kernel_type = kernel
        self.sigma = sigma
        self.degree = degree
        self.kernel = self._get_kernel(kernel)

    def _get_kernel(self, kernel_type):
        if kernel_type == 'rbf':
            return lambda x, y: rbf_kernel(x, y, self.sigma)
        elif kernel_type == 'poly':
            return lambda x, y: poly_kernel(x, y, self.degree)
        elif kernel_type == 'poly_rev':
            return lambda x, y: poly_kernel_reverse(x, y, self.degree)
        elif kernel_type == 'linear':
            return lambda x, y: linear_kernel(x, y)
        else:
            raise ValueError("Unsupported kernel.")

    def fit(self, X, y):
        n_samples, _ = X.shape
        K = np.zeros((n_samples, n_samples))

        # Compute kernel matrix
        for i in range(n_samples):
            for j in range(n_samples):
                K[i, j] = self.kernel(X[i], X[j])

        # QP problem matrices
        P = matrix(np.outer(y, y) * K)
        q = matrix(-np.ones(n_samples))
        A = matrix(y.reshape(1, -1).astype('double'))
        b = matrix(np.zeros(1))

        # Choose G and h based on whether C is None
        if self.C is None:
            # Hard margin: alpha_i ≥ 0
            G = matrix(-np.eye(n_samples))
            h = matrix(np.zeros(n_samples))
        else:
            # Soft margin: 0 ≤ alpha_i ≤ C
            G = matrix(np.vstack((
                -np.eye(n_samples),           # -alpha_i ≤ 0 → alpha_i ≥ 0
                 np.eye(n_samples)            # alpha_i ≤ C
            )))
            h = matrix(np.hstack((
                np.zeros(n_samples),
                np.ones(n_samples) * self.C
            )))

        # Solve QP problem
        solvers.options['show_progress'] = False
        solution = solvers.qp(P, q, G, h, A, b)
        alphas = np.ravel(solution['x'])

        # Support vectors have non-zero alphas
        sv = alphas > 1e-5
        self.alphas = alphas[sv]
        self.support_vectors = X[sv]
        self.support_vector_labels = y[sv]

        # Bias term: averaged over support vectors
        self.b = np.mean([
            y_k - np.sum(self.alphas * self.support_vector_labels *
                         np.array([self.kernel(x_k, x_j) for x_j in self.support_vectors]))
            for (x_k, y_k) in zip(self.support_vectors, self.support_vector_labels)
        ])

    def project(self, X):
        y_pred = []
        for x in X:
            result = np.sum([
                a * y_sv * self.kernel(x, x_sv)
                for a, y_sv, x_sv in zip(self.alphas, self.support_vector_labels, self.support_vectors)
            ])
            y_pred.append(result + self.b)
        return np.array(y_pred)

    def predict(self, X):
        return np.sign(self.project(X))


# SVM + Boosting cvxopt

In [ ]:
class AdaBoostWithSVM:
    def __init__(self, n_estimators=30, C=1.0, kernel='linear', sigma=1.0, degree=3):
        self.n_estimators = n_estimators
        self.C = C
        self.kernel = kernel
        self.sigma = sigma
        self.degree = degree
        self.models = []
        self.alphas = []
        self.train_acc_history = []
        self.test_acc_history = []
        self.X_test = None
        self.y_test = None

    def fit(self, X, y):
        n_samples = X.shape[0]
        w = np.ones(n_samples) / n_samples

        for t in range(self.n_estimators):
            # Sample with probability proportional to weights
            indices = np.random.choice(np.arange(n_samples), size=n_samples, replace=True, p=w)
            X_sample = X[indices]
            y_sample = y[indices]

            # Train SVM (linear)
            clf = KernelSVM(C=self.C, kernel=self.kernel, sigma=self.sigma, degree=self.degree)
            clf.fit(X_sample, y_sample)

            # Predict on full training set
            y_pred = clf.predict(X)

            # Compute weighted error
            incorrect = (y_pred != y).astype(float)
            epsilon = np.sum(w * incorrect)

            if epsilon <= 1e-10 or epsilon >= 0.5:
                continue  # Skip bad learners

            alpha = 0.5 * np.log((1 - epsilon) / epsilon)
            self.models.append(clf)
            self.alphas.append(alpha)

            # Update weights
            w = w * np.exp(-alpha * y * y_pred)
            w = w / np.sum(w)

            # Track train accuracy
            train_pred = self._ensemble_predict(X, upto=len(self.models))
            train_acc = np.mean(train_pred == y)
            self.train_acc_history.append(train_acc)

            # Track test accuracy (if set)
            if self.X_test is not None and self.y_test is not None:
                test_pred = self._ensemble_predict(self.X_test, upto=len(self.models))
                test_acc = np.mean(test_pred == self.y_test)
                self.test_acc_history.append(test_acc)

    def _ensemble_predict(self, X, upto=None):
        upto = upto or len(self.models)
        final_pred = np.zeros(X.shape[0])
        for i in range(upto):
            pred = self.models[i].predict(X)
            final_pred += self.alphas[i] * pred
        return np.sign(final_pred)

    def predict(self, X):
        return self._ensemble_predict(X)


In [ ]:
train_set, train_labels, test_set, test_labels = separate_train_test_manual(
    data=flattened_data,          # <- Or mda_result or flattened_data
    labels=person_labels,     # <- Label array (1D)
    sub_num_total=200,        # <- Total number of subjects
    train_num=200,            # <- Total samples to use for training (adjust if needed)
    task_num=2                # <- Use 1 for identity, 2 for expression classification
)


In [ ]:
# ========== Convert labels to binary {-1, +1} ==========
# Assume neutral = 0, expression = 1
# If not already binary, modify according to your dataset
unique_labels = np.unique(train_labels)
assert len(unique_labels) == 2, "Expected binary classification labels."

label_map = {unique_labels[0]: -1, unique_labels[1]: +1}
train_labels_bin = np.vectorize(label_map.get)(train_labels)
test_labels_bin = np.vectorize(label_map.get)(test_labels)

# ========== Boosted SVM (Linear Kernel) ==========
boosted = AdaBoostWithSVM(n_estimators=20, C=1.0, kernel='linear')
boosted.X_test = test_set
boosted.y_test = test_labels_bin
boosted.fit(train_set, train_labels_bin)


# ========== Evaluate ==========
train_pred = boosted.predict(train_set)
test_pred = boosted.predict(test_set)

train_acc = np.mean(train_pred == train_labels_bin) * 100
test_acc = np.mean(test_pred == test_labels_bin) * 100

print(f"Train Accuracy: {train_acc:.2f}%")
print(f"Test Accuracy:  {test_acc:.2f}%")
print(f"Boosting completed with {len(boosted.models)} rounds.")

In [ ]:
import matplotlib.pyplot as plt

rounds = np.arange(1, len(boosted.train_acc_history) + 1)
plt.plot(rounds, np.array(boosted.train_acc_history) * 100, label="Train Accuracy", marker='o')
plt.plot(rounds, np.array(boosted.test_acc_history) * 100, label="Test Accuracy", marker='s')
plt.xlabel("Boosting Round")
plt.ylabel("Accuracy (%)")
plt.title("AdaBoost with SVM: Accuracy over Rounds")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# ====== Prepare Data ======
train_set, train_labels, test_set, test_labels = separate_train_test_manual(
    data=flattened_data,
    labels=person_labels,
    sub_num_total=200,
    train_num=200,
    task_num=2
)

# Convert labels to binary {-1, +1}
unique_labels = np.unique(train_labels)
assert len(unique_labels) == 2, "Expected binary classification labels."
label_map = {unique_labels[0]: -1, unique_labels[1]: +1}
train_labels_bin = np.vectorize(label_map.get)(train_labels)
test_labels_bin = np.vectorize(label_map.get)(test_labels)

# ====== PCA + AdaBoost with CVXOPT SVM ======
pca_components_list = [1, 2, 3, 10, 50, 80, 100, 120, 139]
pca_adaboost_accuracies = []

for num_components in pca_components_list:
    pca_result_train, components, _, mean_vec = compute_pca(train_set, num_components)
    centered_test = test_set - mean_vec
    pca_result_test = np.dot(centered_test, components)

    boosted = AdaBoostWithSVM(
        n_estimators=20,
        C=1.0,
        kernel='linear'
    )
    boosted.X_test = pca_result_test
    boosted.y_test = test_labels_bin
    boosted.fit(pca_result_train, train_labels_bin)

    test_pred = boosted.predict(pca_result_test)
    test_acc = np.mean(test_pred == test_labels_bin) * 100
    pca_adaboost_accuracies.append(test_acc)

    print(f"PCA={num_components} → Test Accuracy: {test_acc:.2f}%")

# ====== MDA + AdaBoost with CVXOPT SVM ======
mda_components_list = [1]
mda_adaboost_accuracies = []

for num_components in mda_components_list:
    mda_result_train, components, _, mean_vec = compute_mda(train_set, train_labels_bin, num_components)
    centered_test = test_set - mean_vec
    mda_result_test = np.dot(centered_test, components)

    boosted = AdaBoostWithSVM(
        n_estimators=20,
        C=1.0,
        kernel='linear'
    )
    boosted.X_test = mda_result_test
    boosted.y_test = test_labels_bin
    boosted.fit(mda_result_train, train_labels_bin)

    test_pred = boosted.predict(mda_result_test)
    test_acc = np.mean(test_pred == test_labels_bin) * 100
    mda_adaboost_accuracies.append(test_acc)

    print(f"MDA={num_components} → Test Accuracy: {test_acc:.2f}%")

# ====== Plot Results ======
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(pca_components_list, pca_adaboost_accuracies, marker='o', label='PCA + AdaBoost (CVXOPT)')
plt.axhline(mda_adaboost_accuracies[0], linestyle='--', color='orange',
            label=f'MDA + AdaBoost (CVXOPT) ({mda_adaboost_accuracies[0]:.2f}%)')

plt.xlabel("Number of Components")
plt.ylabel("Test Accuracy (%)")
plt.title("AdaBoost (CVXOPT SVM) with PCA and MDA")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
